# Setup `mailroom-ia` — pas à pas

Ce notebook **déploie le projet de A à Z** dans ton Resource Group. Tu exécutes les cellules une par une, en lisant les explications avant chaque cellule.

## Vue d'ensemble (5 min de lecture)

L'application `mailroom-ia` reçoit des PDFs scannés et les classe automatiquement avec l'IA. Deux containers tournent dans Azure :

- **`web`** — site Next.js où l'admin upload les PDFs. Le PDF est posé dans un **Blob Storage**, puis un message est mis dans une **Queue**.
- **`worker`** — script Python déclenché par la queue (KEDA). Il télécharge le PDF, appelle **Document Intelligence** pour l'OCR, puis **Foundry / gpt-5-mini** pour comprendre de quoi parle le document, et enregistre le résultat dans **Cosmos DB**.

Schéma complet : ouvre [docs/architecture.drawio](docs/architecture.drawio) dans VS Code (extension *Draw.io Integration*) ou sur https://app.diagrams.net.

## Ce qu'on va faire dans ce notebook

| § | Étape | Outil | Durée |
|---|-------|-------|------:|
| 1 | Vérifier les outils installés | `az`, `docker`, `bicep` | 1 min |
| 2 | Se connecter à Azure | `az login` | 1 min |
| 3 | Définir les variables (1 seule à éditer) | Python | 1 min |
| 4 | Déployer l'infra **de base** (Bicep #1) | LAW, App Insights, ACR, Storage, Cosmos, Foundry, ACA Env | 8 min |
| 5 | Créer le projet Foundry | `az cli` | 30 s |
| 6 | Builder + pousser les images Docker | ACR Tasks (cloud build) | 6 min |
| 7 | Déployer les **apps** (Bicep #2) | web + worker + RBAC | 1 min |
| 8 | Tester un upload de bout en bout | curl | 3 min |
| 9 | Nettoyage ciblé (optionnel) | `az` | — |

⏱️ **Total : ~25 min** la première fois. Les redéploiements suivants : ~2-3 min.

## Réseau & sécurité — note importante

**Pour ce workshop, toutes les ressources sont en accès public (`publicNetworkAccess: Enabled`).** Pas de VNet, pas de Private Endpoint. C'est volontairement simple : tu te concentres sur l'IA et le code, pas sur la plomberie réseau.

> 🛡️ Le tenant a une **policy** qui force `publicNetworkAccess: Disabled` sur les services de données (Storage, Cosmos, Foundry) **sauf** si la ressource porte le tag `SecurityControl=ignore`. Ce tag est déjà mis automatiquement par nos templates Bicep — tu n'as rien à faire.
>
> Dans une vraie prod, on remplacerait ça par des **Private Endpoints** + une **VNet** + un **Application Gateway** devant le `web`. Hors scope ici.

## RBAC — qui a le droit de faire quoi

Chaque container a sa propre **Managed Identity** (UAMI = User-Assigned Managed Identity). Pas de mot de passe, pas de clé, Azure gère les tokens.

| Identité | Rôles | Sur quoi |
|----------|-------|----------|
| `uami-web-<id>` | AcrPull | ACR (puller l'image) |
| | Storage Blob Data Contributor | Storage (écrire le PDF) |
| | Storage Queue Data Sender | Storage (poster un message) |
| | Cosmos DB Built-in Data Contributor | Cosmos (lire/écrire documents) |
| `uami-worker-<id>` | AcrPull | ACR |
| | Storage Blob Data Contributor | Storage (télécharger + déplacer) |
| | Storage Queue Data Reader + Message Processor | Storage (KEDA + consommer messages) |
| | Cognitive Services User + OpenAI User | Foundry (appeler DI + LLM) |
| | Cosmos DB Built-in Data Contributor | Cosmos |

Tout est défini dans [infra/infra-apps.bicep](infra/infra-apps.bicep) — pas besoin de l'exécuter à la main.

> ⚠️ **Pré-requis** : avoir suivi les notebooks 01 → 07 du parcours. En particulier comprendre Docker / Container Apps (notebook 07).

> 💰 Coût en idle : ~5-10 €/mois (1 replica web min). Worker = 0 € quand la queue est vide. Nettoyage complet = 0 €.

## 0. Les services Azure utilisés — lecture obligatoire (15 min)

Avant de cliquer sur quoi que ce soit, prends le temps de comprendre **ce que fait chaque service**, **pourquoi on l'a choisi**, et **comment les pièces s'emboîtent**. Sans ça, tu vas être perdu dès le premier Bicep.

Cette section est dense — relis-la une deuxième fois après avoir fait le déploiement, ça fera tilt.

---

### A. Compute — où le code tourne

#### 🏗️ Azure Container Apps (ACA)

ACA est l'**hébergeur de containers managed** d'Azure. L'idée : tu donnes une **image Docker** (un .zip de ton appli + son OS) et Azure s'occupe du reste. Pas besoin de comprendre Kubernetes (k8s) — ACA est bâti dessus mais cache toute la complexité.

**Trois objets ACA importants :**

| Objet | Quand l'utiliser | Exemple chez nous |
|-------|------------------|-------------------|
| **Container App Environment** | Un seul, partagé. C'est le "cluster" : réseau, logs, certificats sont mutualisés. | `aca-stage-<id>` |
| **Container App** | Service **long-running** : web API, frontend, daemon. Ingress HTTPS gratuit, scaling auto 1→N replicas selon HTTP/CPU. | `web` (Next.js, port 3000) |
| **Container App Job** | Tâche **ponctuelle** : tourne, fait son boulot, s'éteint. Trois types de trigger : `Manual`, `Schedule` (cron), `Event` (KEDA). | `worker` (`Event` + queue) |

**Pourquoi ACA plutôt que VMs ou AKS ?**
- VMs : il faut tout gérer (OS, patches, scaling, HTTPS) — trop bas niveau pour un POC.
- AKS (Kubernetes natif) : puissant mais courbe d'apprentissage énorme — overkill pour 2 containers.
- ACA : sweet spot. Tu écris ton Dockerfile et 50 lignes de Bicep, tu as la prod.

**Pourquoi ACA plutôt qu'App Service ?**
- App Service est centré sur les apps web (Python/Node/PHP/.NET). ACA est centré containers et **supporte les Jobs event-driven**, ce qu'App Service ne sait pas faire.

**Coût** : facturé à la **vCPU-seconde + Go-seconde**. Idle = quasi gratuit (consumption plan).

---

#### ⚡ KEDA (Kubernetes Event-Driven Autoscaling)

KEDA, c'est le **chef d'orchestre du scaling** dans ACA. Il est livré "dans la boîte" — tu n'as rien à installer. Son rôle : regarder une **source d'événements externe** (queue, topic, base, cron, custom HTTP) et **calculer combien de replicas/jobs tu devrais avoir maintenant**.

**Catalogue de scalers KEDA** (40+ disponibles) : `azure-queue`, `azure-servicebus`, `azure-eventhub`, `azure-blob`, `kafka`, `redis`, `cron`, `cpu`, `memory`, `http`, etc.

**Notre cas : scaler `azure-queue`**
```yaml
type: azure-queue
metadata:
  accountName: stmailroom<id>     # ton storage account
  queueName: doc-to-classify       # ta queue
  queueLength: "1"                 # 1 replica par message
identity: workerUami               # KEDA poll avec l'identité du worker
```

**Comportement concret :**
1. KEDA poll la queue toutes les 30 s (`pollingInterval`).
2. Si `messages_in_queue >= 1` → KEDA lance une **execution** du Job worker.
3. Le worker démarre (~5 s cold start), prend le message, fait son boulot, s'éteint.
4. Si toujours des messages → KEDA lance d'autres executions en parallèle (jusqu'à `maxExecutions: 10` chez nous).
5. Queue vide → plus rien ne tourne → **0 € de compute**.

**Pourquoi KEDA et pas Azure Functions ?**
- Functions est plus simple **si** tu acceptes ses contraintes : runtime imposé (Python 3.11, Node 22…), pas de Dockerfile custom, dépendances OS limitées (pas de `apt-get install poppler-utils` par exemple).
- Notre worker a besoin du SDK `azure-ai-documentintelligence` + `azure-ai-projects` + `openai` + `aiohttp` → empilement Python lourd. ACA Job + KEDA donne **tu contrôles l'image**, donc tu installes ce que tu veux.

---

#### 🐳 Azure Container Registry (ACR)

C'est ton **Docker Hub privé**. Tu y pushes tes images Docker, ACA les pull pour les exécuter.

**Pourquoi pas Docker Hub public ?**
- Sécurité : pas envie que ton image soit accessible au monde.
- Compliance : les images Azure doivent rester dans Azure (DLP, audit).
- Performance : pull intra-Azure ~10× plus rapide qu'un pull depuis Internet.

**Fonction bonus : `az acr build`**

C'est LA killer feature pour les stagiaires. Tu envoies ton dossier source (avec le Dockerfile) à ACR via `az acr build` → ACR builde l'image **sur ses propres VMs Linux** et la stocke directement. Tu n'as **pas besoin de Docker actif** sur ton poste.

Limites :
- Pas de support BuildKit (donc pas de `RUN --mount=type=cache` dans le Dockerfile).
- Plus lent qu'un build local Docker en itératif (~3-5 min pour notre web Next.js).

---

### B. Stockage — où les données vivent

#### 💾 Azure Storage Account

Un compte de stockage Azure, c'est **4 services différents derrière un seul endpoint**. C'est confus au début. On utilise les 2 premiers :

| Sous-service | Endpoint | Pour quoi faire | Notre usage |
|--------------|----------|-----------------|-------------|
| **Blob** | `https://<acct>.blob.core.windows.net/` | Stocker des fichiers (jusqu'à 5 TB par blob) | PDFs uploadés → `mailroom/_inbox/<uuid>.pdf` ; après classification → `mailroom/clients/<clientId>/<categ>/<file>.pdf` |
| **Queue** | `https://<acct>.queue.core.windows.net/` | File d'attente FIFO simple, 64 KB max par message, rétention 7 jours par défaut | `doc-to-classify` : `web` poste, `worker` consomme |
| Table | `.table…` | NoSQL clé/valeur (pré-Cosmos) | (pas utilisé) |
| File | `.file…` | Partage SMB/NFS | (pas utilisé) |

**Concepts Blob importants :**
- **Container** (ne pas confondre avec Docker container) = "dossier racine" dans ton Storage. Le nôtre s'appelle `mailroom`.
- **Blob name** = chemin virtuel avec `/`. C'est juste une convention — il n'y a pas de vrais dossiers, mais les outils affichent une arborescence.
- **Metadata** = clés/valeurs custom attachées à un blob (`x-ms-meta-id`, `x-ms-meta-originalname`…).
- **Tier** = `Hot` (accès fréquent, cher au Go), `Cool`, `Archive` (rare, ultra cheap). Pas pertinent pour notre POC.

**Concepts Queue importants :**
- **Message** = blob de bytes (souvent JSON), encodé en base64 par défaut.
- **Visibility timeout** = quand un consommateur lit un message, il devient invisible pour les autres pendant N secondes. S'il ne le `delete` pas dans ce délai, le message redevient visible → réessai automatique. Pratique pour le retry, dangereux si tu fais des trucs non-idempotents.
- **Dequeue count** = combien de fois ce message a été lu sans être deleté. Au-delà d'un seuil (configurable), KEDA/ton code peut le considérer comme "poison".

> 🤔 **Comparaison messaging Azure** (à connaître pour les entretiens) :
>
> | Service | Best for | Coût | Garanties |
> |---------|----------|------|-----------|
> | **Storage Queue** | POC, volume modéré, message court | Quasi gratuit | At-least-once, FIFO best-effort |
> | **Service Bus** | Entreprise — transactions, sessions, dead-letter, pub/sub | $$ | At-least-once strict, FIFO garanti par session, ordre, schema |
> | **Event Hubs** | Stream haut débit (millions d'evts/s, telemetry, IoT) | $$$ | Append-only log, replay possible |
> | **Event Grid** | Réagir à des **événements** Azure ou custom (pub/sub léger) | $ | At-least-once, filtrage côté broker |
>
> Chez nous : `web` connaît le `worker` et veut lui parler 1-pour-1 → **queue** suffit. **Storage Queue** = le moins cher / le plus simple → ✅.
>
> Quand utiliserait-on Event Grid ici ? Imagine qu'un fournisseur dépose un PDF dans notre Blob via SFTP. On ne contrôle pas le code du fournisseur, donc personne ne peut "poster un message dans la queue". Solution : `BlobCreated` → Event Grid → handler qui pose un message dans la queue. Pattern hyper courant.

---

#### 🗄️ Azure Cosmos DB (NoSQL, mode serverless)

Cosmos est la **base NoSQL managed d'Azure**. C'est multi-modèle (SQL/MongoDB/Cassandra/Gremlin), mais on utilise l'API **SQL/NoSQL** (la plus courante, la plus moderne).

**Concepts à connaître :**

- **Account** = ressource Azure facturée (ex. `cosmos-stage-<id>`).
- **Database** = groupe logique de containers (ex. `mailroom`).
- **Container** (encore un mot surchargé 😅) = une "table" Cosmos. Chez nous : `documents` et `clients`.
- **Item** = un document JSON. Doit avoir un champ `id` unique dans sa partition.
- **Partition key** = la clé qui détermine **sur quelle machine physique** un item est stocké. Choix critique pour la perf.
  - `documents` : partition `/clientId` (les docs d'un même client sont colocalisés → query rapide).
  - `clients` : partition `/id` (un client = sa propre mini-partition, on les query un par un).
- **Request Unit (RU)** = l'unité de coût Cosmos. Une lecture par id ≈ 1 RU ; une query cross-partition = 5-50 RU ; une écriture ≈ 5-10 RU.

**Mode `serverless` (le nôtre)** :
- Tu payes **par RU consommée**, pas à l'heure.
- Pas de RU/s à réserver.
- Limite : 1 To max par container, throughput non garanti au-delà de 5000 RU/s.
- Pour un POC c'est parfait, ~quelques centimes par mois si tu testes 20 docs.

**L'autre mode** est `provisioned throughput` (tu réserves N RU/s 24/7, plus cher mais plus prévisible). Pour la prod à fort trafic.

> 🔐 **RBAC Cosmos data-plane** (point qui surprend tout le monde) :
> Azure RBAC (`Cosmos DB Account Reader`, `Cosmos DB Contributor`…) donne accès au **management plane** : créer un compte, lister les databases, voir les métriques. **Il NE donne PAS le droit de lire ou écrire des items.**
>
> Pour lire/écrire dans Cosmos, il faut un **rôle Cosmos natif** assigné via `sqlRoleAssignments`. Le rôle built-in **`Cosmos DB Built-in Data Contributor`** (id `00000000-0000-0000-0000-000000000002`) donne CRUD complet. C'est pour ça qu'on a une étape §7.5 dédiée.

---

### C. Intelligence Artificielle

#### 🧠 Microsoft Foundry (anciennement Azure AI Studio)

Foundry est la **plateforme IA unifiée** d'Azure. Une ressource Azure de `kind=AIServices` te donne accès, **derrière un même endpoint et avec la même Managed Identity**, à plein de services IA :

- **Azure OpenAI** — déploiement de mod`èles GPT (chat completion, embeddings, fine-tuning).
- **Document Intelligence** (anciennement Form Recognizer) — OCR + extraction structurée. Modèles prebuilt (`layout`, `invoice`, `receipt`, `id-document`…) + tu peux entraîner tes custom.
- **Azure AI Vision** — analyse d'images.
- **Azure AI Speech** — STT/TTS.
- **Translator**.
- **Content Safety** — modération.
- **Foundry Agents, Evaluations, Tracing** — plateforme pour build/évaluer des agents (hors scope ici).

**Hiérarchie Foundry :**
```
AIServices account  (notre aif-stage-<id>)
 ├── Model deployments  (gpt-5-mini, text-embedding-3-small…)
 ├── Custom subdomain   (obligatoire pour auth Entra ID)
 └── Projects           (workspaces isolés ; notre "mailroom-project")
      ├── Agents
      ├── Datasets
      ├── Evaluations
      └── Indexes (AI Search vector)
```

**Comment on l'utilise :**
- **Worker → Document Intelligence** : `POST /documentintelligence/documentModels/prebuilt-layout:analyze` avec le PDF en bytes → renvoie le texte OCRisé + la structure (tableaux, paragraphes).
- **Worker → gpt-5-mini** : `POST /openai/v1/chat/completions` avec le texte OCR + la liste des clients connus → renvoie une classification structurée (Pydantic) `{client_id, category, confidence, reasoning}`.

Les deux appels passent par le **même endpoint Foundry** (`https://aif-stage-<id>.cognitiveservices.azure.com/`) avec le même token Entra ID. Pas deux ressources à gérer, pas deux clés à protéger.

---

### D. Observabilité — où on regarde quand ça casse

#### 📊 Log Analytics Workspace (LAW)

LAW est le **data warehouse de logs/métriques** d'Azure. Quasi tous les services Azure y poussent des données. Tu interroges en **KQL** (Kusto Query Language) — syntaxe pipeline à la Splunk/Sentinel.

**Tables auto-créées par ACA :**
- `ContainerAppConsoleLogs_CL` : `stdout`/`stderr` de tes containers (1 ligne = 1 ligne de log).
- `ContainerAppSystemLogs_CL` : événements de la plateforme (`ImagePulled`, `ContainerStarted`, `BackoffLimitExceeded`…).

**Exemple KQL** :
```kusto
ContainerAppConsoleLogs_CL
| where TimeGenerated > ago(15m)
| where ContainerName_s == 'worker'
| where Log_s contains 'error'
| project TimeGenerated, Log_s
| order by TimeGenerated desc
```

**Coût** : ~2 €/Go ingéré. Pour un POC ça reste négligeable, mais en prod attention aux logs verbeux.

#### 📈 Application Insights

App Insights est le SDK de **tracing applicatif** (compatible OpenTelemetry). Là où LAW stocke tes `print()`, App Insights stocke des **spans** structurés (start time, end time, parent span id, attributes).

Concrètement : le worker, avec `azure-monitor-opentelemetry`, va envoyer :
```
trace_id=abc
  span "process_document" (24 s)
    span "blob.download" (200 ms)
    span "di.extract" (8 s)
    span "llm.classify" (12 s)
    span "cosmos.upsert" (50 ms)
```

Tu visualises ça dans le portail Azure → App Insights → "Application Map" / "End-to-end transaction". Très utile pour comprendre où ton agent passe son temps. Cf. notebook 06.

---

### E. Identité & sécurité

#### 🆔 Managed Identity (MI)

**Le problème** : ton worker doit prouver à Azure qu'il a le droit d'écrire dans le Blob. Solution naïve : une connection string en variable d'environnement. Problème : la string traîne en clair dans les configs, dans les logs, dans git si tu fais pas gaffe.

**La solution Managed Identity** : Azure crée une identité Entra ID pour ta ressource (container, VM, function…). Elle n'a pas de mot de passe ; à la place, **Azure injecte un endpoint local** (`http://localhost:12356/msi/token` dans ACA) que ton code interroge pour obtenir un token Entra ID. Le SDK Azure (`DefaultAzureCredential` en .NET/Python/JS) fait ça automatiquement.

Deux types :
| Type | Cycle de vie | Quand l'utiliser |
|------|--------------|------------------|
| **System-Assigned (SAMI)** | Créée avec la ressource, supprimée avec elle. 1:1. | Simple, identité jetable. |
| **User-Assigned (UAMI)** | Ressource Azure indépendante (`Microsoft.ManagedIdentity/userAssignedIdentities`). Peut être attachée à plusieurs ressources. | Quand tu as besoin de l'identité **avant** ou **après** la ressource consommatrice. |

**Pourquoi UAMI chez nous ?** Parce que le rôle `AcrPull` doit exister AVANT que le container démarre — sinon il échoue à pull l'image et l'opération ARM timeout ("Operation expired"). Avec UAMI on crée l'identité + on l'autorise sur l'ACR dans un premier `dependsOn`, et seulement APRÈS on crée le container.

> 🔑 Côté code Python (worker) : `cred = DefaultAzureCredential()` lit `AZURE_CLIENT_ID` de l'env (qu'on injecte depuis le Bicep) pour choisir LA bonne UAMI, demande un token au endpoint local, et le SDK Blob/Cosmos/Foundry l'utilise pour signer les requêtes. **Zéro secret en clair.**

#### 🛡️ Azure Policy

Azure Policy = des **règles obligatoires** qu'une organisation impose sur sa subscription / management group. Trois types d'effet courants :
- `Audit` : log mais n'empêche rien (juste pour mesurer la conformité).
- `Deny` : bloque la création de la ressource si non-conforme.
- `Modify` : ajoute/modifie automatiquement un champ (ex. tag).

**Chez nous** : le tenant a une policy qui force `publicNetworkAccess: Disabled` sur les services de données (Storage, Cosmos, Foundry…) **sauf** si la ressource porte le tag `SecurityControl=ignore`. Ce tag est dans nos templates Bicep → toutes nos ressources le portent à la création → on garde l'accès public (acceptable pour ce workshop).

En prod, on **n'utiliserait pas** ce tag — on mettrait en place des Private Endpoints à la place. Hors scope ici.

---

### Le chemin d'un PDF, étape par étape

```
[Admin]
  │ 1. Upload PDF via UI
  ▼
[web Container App]  (ACA, ingress public HTTPS)
  │ 2. Auth Managed Identity → token Entra ID
  │ 3. PUT blob mailroom/_inbox/<uuid>.pdf            (Storage Blob)
  │ 4. Enqueue JSON {id, blobName, ...}               (Storage Queue)
  │ ✓ Retourne 200 à l'admin
  ▼
[Storage Queue: doc-to-classify]  ─ message attend
  │
  │ 5. KEDA scaler "azure-queue" poll toutes les 30 s
  │    voit messages_in_queue >= 1, déclenche une exécution
  ▼
[worker Container App Job]  (ACA, démarrage à froid ~5 s)
  │ 6. Auth Managed Identity (UAMI dédiée)
  │ 7. Read message from queue
  │ 8. Download blob bytes                            (Storage Blob)
  │ 9. POST bytes → prebuilt-layout                   (Foundry / Document Intelligence)
  │      → texte OCR
  │ 10. POST chat.completions.parse(...)              (Foundry / gpt-5-mini)
  │      → {client_id, category, confidence}
  │ 11. (si confidence haute) move blob _inbox → clients/<clientId>/<categ>/...
  │ 12. Upsert document JSON                          (Cosmos DB)
  │ 13. Delete message                                (Storage Queue)
  └─ Container s'éteint (jusqu'au prochain message)

  Tout au long : stdout/stderr → ContainerAppConsoleLogs_CL (LAW)
                 spans OpenTelemetry → App Insights
```

⚠️ **Réseau (résumé)** : tous ces services sont **en accès public**, protégés par leur firewall identité (Entra ID/RBAC). Pas de VNet, pas de Private Endpoint, pas d'App Gateway. En prod on isolerait — mais pas le sujet ici.

## 1. Vérifier que les outils sont installés

On a besoin de trois outils :

- **Azure CLI** (`az`) : pour piloter Azure depuis la ligne de commande.
- **Bicep CLI** : pour déployer l'infrastructure (livré avec az).
- **Docker** : juste pour la commande `docker --version`. **Tu n'as PAS besoin de Docker Desktop actif** — on builde les images dans le cloud avec `az acr build` plus tard.

Si une commande échoue : (re)installe l'outil et relance.

In [ ]:
# Vérification
!az --version
!az bicep version
!docker --version

In [ ]:
# Installer l'extension Azure CLI pour Container Apps (idempotent — ne fait rien si déjà installée).
# Puis enregistrer les "resource providers" : c'est l'opération qui dit à ta subscription
# "j'autorise la création de ressources de tel type". À faire UNE fois par subscription.
!az extension add --name containerapp --upgrade --only-show-errors
!az provider register --namespace Microsoft.App --wait
!az provider register --namespace Microsoft.OperationalInsights --wait
!az provider register --namespace Microsoft.CognitiveServices --wait
!az provider register --namespace Microsoft.DocumentDB --wait
!az provider register --namespace Microsoft.ManagedIdentity --wait

## 2. Se connecter à Azure

Avant TOUTE commande `az` qui touche aux ressources, il faut s'authentifier.


- Sur un poste perso : `az login` ouvre un navigateur.Après connexion, vérifie que tu es sur la **bonne subscription** (celle de ton workshop, pas une perso).

- Dans un Codespace / SSH : `az login --use-device-code` (ça affiche un code à coller sur https://microsoft.com/devicelogin).

In [ ]:
# Décommente la première ligne si pas encore connecté, puis relance.
# !az login --use-device-code

# Si tu as plusieurs subscriptions, choisis-en une :
# !az account set --subscription "<nom-ou-id>"

!az account show --output table

## 3. Définir les variables

✨ **Une seule variable à éditer dans la cellule suivante** : `RG` = le nom de ton Resource Group (format `rg-stage-<ton-identifiant>`).

Tous les autres noms (ACR, Storage, Cosmos, Foundry, etc.) sont **calculés automatiquement** à partir de ton identifiant. Ça garantit que deux stagiaires n'ont pas de collision de noms (les noms de Storage Account et ACR doivent être uniques au monde 👋).

La cellule crée aussi une petite fonction `az()` qui rend les erreurs lisibles — sinon les messages d'`az cli` sont vite illisibles.

In [ ]:
import os, re, subprocess
from pathlib import Path

# 👇 SEULE VARIABLE À ÉDITER
RG = "rg-stage-sadk"

m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")


def az(cmd: str) -> str:
    """Lance une commande az et renvoie stdout. Affiche stderr lisible en cas d'échec."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"❌ Commande échouée (exit {result.returncode}):\n"
            f"   $ {cmd}\n"
            f"   stderr: {result.stderr.strip() or '(vide)'}\n"
            f"   stdout: {result.stdout.strip() or '(vide)'}\n"
            f"   👉 Vérifiez : (1) vous êtes connecté (`az login`), "
            f"(2) bonne subscription active (`az account show`), "
            f"(3) le RG '{RG}' existe et vous y avez Owner/Contributor."
        )
    return result.stdout.strip()


LOCATION = az(f"az group show --name {RG} --query location -o tsv")

# Chemin racine du projet (où on se trouve)
PROJECT_DIR = Path.cwd()
INFRA_DIR = PROJECT_DIR / "infra"
WEB_DIR = PROJECT_DIR / "web"
WORKER_DIR = PROJECT_DIR / "worker"

# Noms dérivés (alignés sur les Bicep)
clean = USER_ID.replace("-", "")
ACR_NAME = f"acrmailroom{clean}"
ACR_LOGIN = f"{ACR_NAME}.azurecr.io"
STORAGE_NAME = f"stmailroom{clean}"
COSMOS_NAME = f"cosmos-stage-{USER_ID}"
FOUNDRY_NAME = f"aif-stage-{USER_ID}"
FOUNDRY_PROJECT = "mailroom-project"
ACA_ENV = f"aca-stage-{USER_ID}"
WEB_APP = f"web-stage-{USER_ID}"
WORKER_JOB = f"worker-stage-{USER_ID}"

for k, v in {
    "RG": RG, "USER_ID": USER_ID, "LOCATION": LOCATION,
    "ACR_NAME": ACR_NAME, "ACR_LOGIN": ACR_LOGIN, "STORAGE_NAME": STORAGE_NAME,
    "COSMOS_NAME": COSMOS_NAME, "FOUNDRY_NAME": FOUNDRY_NAME,
    "FOUNDRY_PROJECT": FOUNDRY_PROJECT, "ACA_ENV": ACA_ENV,
    "WEB_APP": WEB_APP, "WORKER_JOB": WORKER_JOB,
}.items():
    os.environ[k] = v

print(f"✓ RG trouvé : {RG}  (région : {LOCATION})")
print(f"  Identifiant : {USER_ID}")
print(f"  ACR         : {ACR_LOGIN}")
print(f"  Foundry     : {FOUNDRY_NAME}")
print(f"  ACA env     : {ACA_ENV}")

## 4. Déployer l'infrastructure de base (Bicep #1)

> 💡 **Rappel §0.A/B** : on va créer les ressources de **plomberie** (sans containers encore) : LAW, App Insights, ACR, Storage, Cosmos, Foundry et l'ACA Environment vide. Les Container Apps eux-mêmes viennent en §7 (après qu'on ait pushé les images).

[infra/infra-base.bicep](infra/infra-base.bicep) crée **toutes les ressources "plumbing"** dont on aura besoin avant de pouvoir lancer le code :

| Ressource | À quoi ça sert |
|-----------|----------------|
| Log Analytics Workspace | Stocker tous les logs (web, worker, ACA env) |
| Application Insights | Tracing & métriques applicatives (latence, exceptions…) |
| Azure Container Registry (ACR) | Héberger nos images Docker `mailroom-web` et `mailroom-worker` |
| Storage Account | Blob pour les PDFs + Queue pour le pipeline async |
| Cosmos DB (serverless) | Métadonnées : 1 document par PDF classé, 1 document par client |
| Microsoft Foundry (kind=AIServices) | LLM (gpt-5-mini) + Document Intelligence sur le même endpoint |
| Container Apps Environment | L'environnement de runtime managed-K8s où tournent `web` et `worker` |

⏱️ **~8 min** (Cosmos est le plus lent).

### ⚠️ Deux pièges connus

1. **Cosmos en West Europe peut renvoyer `ServiceUnavailable`** (capacité zonale saturée). On le déploie donc en **France Central** par défaut (param `cosmosLocation`). Le reste reste dans la région du RG.
2. **ACA Environment peut renvoyer `ManagedEnvironmentCapacityHeavyUsageError`** (capacité AKS saturée). Idem — on bascule en France Central si besoin via `acaLocation`.

La latence inter-région West EU ↔ France Central = quelques ms, sans impact pour ce POC.

In [ ]:
# Choix des régions cibles.
#   ACA_LOCATION    : région du Container Apps Environment.
#   COSMOS_LOCATION : région du compte Cosmos.
# On force West Europe + France Central par défaut car c'est la combinaison qui passe
# de manière fiable au moment du workshop.
ACA_LOCATION = "westeurope"
COSMOS_LOCATION = "francecentral"

os.environ["ACA_LOCATION"] = ACA_LOCATION
os.environ["COSMOS_LOCATION"] = COSMOS_LOCATION
print(f"Région du RG     : {LOCATION}")
print(f"ACA Env          : {ACA_LOCATION}")
print(f"Cosmos DB        : {COSMOS_LOCATION}")

### 4.1 (Optionnel) Nettoyer un Cosmos en échec

Si un déploiement précédent s'est arrêté en plein milieu, Cosmos peut rester bloqué dans l'état **`Failed`**. Dans ce cas, ARM refuse de le recréer tant qu'on ne le supprime pas explicitement. La cellule suivante détecte ce cas.

In [ ]:
# Vérifier si Cosmos est en provisioning failed (cas du retry après crash)
cosmos_state = subprocess.run(
    f"az cosmosdb show --name {COSMOS_NAME} --resource-group {RG} "
    f"--query provisioningState -o tsv",
    shell=True, capture_output=True, text=True,
).stdout.strip()

if cosmos_state:
    print(f"Cosmos {COSMOS_NAME} → state: {cosmos_state}")
    if cosmos_state.lower() == "failed":
        print("⚠️  Cosmos est en état 'Failed' — il faut le supprimer avant de relancer la passe 1.")
        print("    Décommentez la ligne ci-dessous pour le supprimer (~2 min) :")
        print(f"    !az cosmosdb delete --name {COSMOS_NAME} --resource-group {RG} --yes")
        # !az cosmosdb delete --name $COSMOS_NAME --resource-group $RG --yes
else:
    print(f"✓ Cosmos {COSMOS_NAME} n'existe pas encore → tout est OK pour déployer.")

In [ ]:
# Validation Bicep — c'est un dry-run : ARM vérifie la syntaxe et les permissions
# SANS rien créer. Si ça passe ici, le déploiement réel a de bonnes chances de marcher.
!az deployment group validate \
    --resource-group $RG \
    --template-file "{INFRA_DIR}/infra-base.bicep" \
    --parameters userId=$USER_ID acaLocation=$ACA_LOCATION cosmosLocation=$COSMOS_LOCATION \
    --output table

In [ ]:
# Déploiement réel de la passe 1. ~8 min.
# Bicep est idempotent : si tu relances, il ne touche que ce qui a changé.
!az deployment group create \
    --name mailroom-base \
    --resource-group $RG \
    --template-file "{INFRA_DIR}/infra-base.bicep" \
    --parameters userId=$USER_ID acaLocation=$ACA_LOCATION cosmosLocation=$COSMOS_LOCATION \
    --output table

In [ ]:
import json

# 1. État du déploiement
state = az(
    f"az deployment group show --resource-group {RG} --name mailroom-base "
    f"--query properties.provisioningState -o tsv"
)
print(f"Déploiement mailroom-base : provisioningState = {state}")

if state.lower() != "succeeded":
    print("\n❌ Le déploiement n'est pas en 'Succeeded' → pas d'outputs à récupérer.")
    print("Détails des opérations en échec :\n")
    # On passe par subprocess (pas !... shell) pour interpoler {RG} correctement
    fails = subprocess.run(
        [
            "az", "deployment", "operation", "group", "list",
            "--name", "mailroom-base",
            "--resource-group", RG,
            "--query",
            "[?properties.provisioningState=='Failed']."
            "{resource:properties.targetResource.resourceName,"
            "type:properties.targetResource.resourceType,"
            "code:properties.statusMessage.error.code,"
            "message:properties.statusMessage.error.message}",
            "-o", "jsonc",
        ],
        shell=True, capture_output=True, text=True,
    )
    print(fails.stdout or fails.stderr)
    raise RuntimeError(
        f"Déploiement en état '{state}'. Voir les détails ci-dessus, "
        f"corriger et relancer la cellule de déploiement."
    )

# 2. Récupérer les outputs (le déploiement est OK)
outputs_json = az(
    f"az deployment group show --resource-group {RG} --name mailroom-base "
    f"--query properties.outputs -o json"
)
if not outputs_json or outputs_json == "null":
    raise RuntimeError(
        "Le déploiement est Succeeded mais ne renvoie aucun output. "
        "Vérifiez que infra-base.bicep contient bien les blocs `output ...`."
    )

OUTPUTS = {k: v["value"] for k, v in json.loads(outputs_json).items()}

print("\nOutputs du déploiement infra-base :\n")
for k, v in OUTPUTS.items():
    print(f"  {k:30s} = {v}")

# Et on les exporte en env pour les cellules shell qui suivent
for k, v in OUTPUTS.items():
    os.environ[k.upper()] = str(v)

## 5. Créer le projet Foundry

> 💡 **Rappel §0.C** : un compte Foundry (`kind=AIServices`) peut héberger plusieurs **projets** (=workspaces). Un projet, c'est ton bac à sable : il a son propre endpoint, ses propres déploiements de modèles, ses agents.

On crée le projet en **REST direct** parce que l'API `accounts/projects` Bicep est en preview et change tout le temps. Plus stable comme ça.

Après cette étape, tu pourras ouvrir [https://ai.azure.com](https://ai.azure.com) et voir ton projet `mailroom-project` avec le modèle `gpt-5-mini` déjà déployé (le modèle, lui, est déployé par Bicep dans la passe 1).

In [ ]:
# (1) On active le "custom subdomain" sur la ressource Foundry. C'est obligatoire
#     pour que les appels Entra ID (token-based auth) fonctionnent.
!az cognitiveservices account update \
    --name $FOUNDRY_NAME --resource-group $RG \
    --custom-domain $FOUNDRY_NAME \
    --output none

# (2) Création du projet Foundry via REST direct.
import json as _json
_sub = az("az account show --query id -o tsv")
_uri = (
    f"https://management.azure.com/subscriptions/{_sub}"
    f"/resourceGroups/{RG}/providers/Microsoft.CognitiveServices"
    f"/accounts/{FOUNDRY_NAME}/projects/{FOUNDRY_PROJECT}?api-version=2025-04-01-preview"
)
_body = {"location": LOCATION, "identity": {"type": "SystemAssigned"}, "properties": {}}
with open("project-body.json", "w", encoding="utf-8") as f:
    f.write(_json.dumps(_body))
!az rest --method PUT --uri "{_uri}" --body "@project-body.json" --query "{{name:name,state:properties.provisioningState}}" -o table

## 6. Builder + pousser les images Docker

> 💡 **Rappel §0.A** : ACR est notre registre Docker privé. ACA en pull les images quand il démarre les containers. `az acr build` builde l'image **dans le cloud** — pas besoin de Docker local actif.

Deux images à construire :
- `mailroom-web:v1` — Next.js (~3-4 min)
- `mailroom-worker:v1` — Python (~2 min)

> 📝 ACR Tasks utilise le builder Docker **legacy** (pas BuildKit) — nos Dockerfiles sont déjà compatibles (pas de `RUN --mount=type=cache`).

In [ ]:
# Build de l'image web (Next.js). ~3-5 min.
# --no-logs : évite un bug d'encodage Windows quand Next.js affiche le triangle ▲.
!az acr build \
    --registry $ACR_NAME \
    --image mailroom-web:v1 \
    --file "{WEB_DIR}/Dockerfile" \
    --no-logs \
    "{WEB_DIR}"

In [ ]:
# Build de l'image worker (Python). ~2 min.
!az acr build \
    --registry $ACR_NAME \
    --image mailroom-worker:v1 \
    --file "{WORKER_DIR}/Dockerfile" \
    --no-logs \
    "{WORKER_DIR}"

In [ ]:
# Vérifier que les 2 images sont bien dans l'ACR.
# On doit voir mailroom-web et mailroom-worker, chacun avec un tag v1.
!az acr repository list --name $ACR_NAME --output table
print()
!az acr repository show-tags --name $ACR_NAME --repository mailroom-web -o table
!az acr repository show-tags --name $ACR_NAME --repository mailroom-worker -o table

## 7. Déployer les apps (Bicep #2)

> 💡 **Rappel §0.A** : on crée maintenant les Container Apps eux-mêmes (`web` = service long-running, `worker` = Job event-driven via KEDA). Avec les **UAMI** (User-Assigned Managed Identities, §0.E) qui leur donnent accès à ACR, Storage, Cosmos et Foundry sans aucune clé.

[infra/infra-apps.bicep](infra/infra-apps.bicep) crée :

- **`web-stage-<id>`** : Container App (Next.js), ingress public HTTPS sur port 3000, 1–3 replicas auto-scalés.
- **`worker-stage-<id>`** : Container App **Job** event-driven via KEDA (`scaler azure-queue`, cf. §0.A). Tourne uniquement quand la queue contient des messages.
- **2 User-Assigned Managed Identities** (UAMI) — une par app, avec tous les rôles nécessaires déjà attribués.

> 📝 **Pourquoi UAMI et pas SystemAssigned ?** (cf. §0.E) Les rôles `AcrPull` doivent exister AVANT que le container ne démarre, sinon il timeout au pull. Avec UAMI, on crée l'identité + l'autorise AVANT de créer le container (via `dependsOn` Bicep).

⏱️ **~1 min**.

In [ ]:
# On reprend la région ACA depuis les outputs de la passe 1
# (sécurité si on a basculé de région entre temps).
ACA_LOCATION_USED = OUTPUTS.get("acaEnvLocation", ACA_LOCATION)
os.environ["ACA_LOCATION_USED"] = ACA_LOCATION_USED
print(f"ACA Env est dans : {ACA_LOCATION_USED}")

!az deployment group create \
    --name mailroom-apps \
    --resource-group $RG \
    --template-file "{INFRA_DIR}/infra-apps.bicep" \
    --parameters userId=$USER_ID acaLocation=$ACA_LOCATION_USED webImageTag=v1 workerImageTag=v1 \
    --output table

In [ ]:
# Récupère l'URL publique de la web app (générée par ACA).
FQDN = subprocess.check_output(
    f"az containerapp show -n {WEB_APP} -g {RG} --query properties.configuration.ingress.fqdn -o tsv",
    shell=True,
).decode().strip()
print(f"🌍 Web app : https://{FQDN}")
print(f"   /admin/inbox    -> page d'upload")
print(f"   /admin/clients  -> liste clients")
print(f"   /admin/storage  -> arborescence blob")

## 7.5 RBAC data-plane Cosmos DB

> 💡 **Rappel §0.B** : Cosmos a son **propre système RBAC** (les `sqlRoleAssignments`), séparé d'Azure RBAC. Azure RBAC contrôle qui peut **gérer le compte** (créer la db, voir les métriques…). Il NE donne PAS le droit de lire/écrire des items.

Pour qu'un UAMI puisse faire `upsert`/`query` sur les containers, on lui assigne le rôle built-in **`Cosmos DB Built-in Data Contributor`** (id `00000000-0000-0000-0000-000000000002`).

Cette étape est aussi dans le Bicep `infra-apps.bicep`, mais on la répète ici par sécurité — c'est **le bug n°1** quand un worker n'arrive pas à écrire dans Cosmos.

In [ ]:
COSMOS_DATA_CONTRIB = "00000000-0000-0000-0000-000000000002"
COSMOS_ID = az(f"az cosmosdb show -n {COSMOS_NAME} -g {RG} --query id -o tsv")

for name in (f"uami-web-{USER_ID}", f"uami-worker-{USER_ID}"):
    principal = az(f"az identity show -n {name} -g {RG} --query principalId -o tsv")
    print(f"Granting Cosmos Data Contributor → {name} ({principal[:8]}…)")
    !az cosmosdb sql role assignment create \
        --account-name $COSMOS_NAME --resource-group $RG \
        --scope "{COSMOS_ID}" \
        --principal-id "{principal}" \
        --role-definition-id "{COSMOS_DATA_CONTRIB}" \
        --output none 2>nul
print("✓ Done.")

## 8. Test de bout en bout

> 💡 **Rappel §0 — flux complet** : voilà là où toutes les pièces qu'on a vues s'emboitent. Le `web` (ACA) reçoit l'upload, pose dans le **Blob** + **Queue** (Storage), **KEDA** détecte le message et lance le `worker` (ACA Job), qui appelle **Document Intelligence** (OCR) puis **Foundry** (LLM) et enregistre dans **Cosmos**. Tout en étant authentifié par **Managed Identity** — zero clé.

Ouvre l'URL ci-dessus dans ton navigateur, va sur **`/admin/inbox`** et upload un PDF. Le flux complet est :

```
1. Browser   --(POST /api/upload)-->   web (Next.js)
2. web       --(PUT blob)----------->   Storage Blob (_inbox/<uuid>.pdf)
3. web       --(enqueue)------------>   Storage Queue (doc-to-classify)
4. KEDA      --(poll 30s)----------->   déclenche une exécution du Job worker
5. worker    --(download)----------->   Storage Blob (lit le PDF en bytes)
6. worker    --(prebuilt-layout)---->   Document Intelligence (OCR)
7. worker    --(chat.completions)--->   Foundry gpt-5-mini (classification structurée)
8. worker    --(upsert)------------->   Cosmos DB (documents container)
```

Durée attendue : ~20-25 s par document.

La cellule suivante vérifie qu'au moins une exécution du job a eu lieu après ton upload.

In [ ]:
# Liste les dernières exécutions du job worker.
# Après un upload, tu devrais voir une nouvelle ligne en "Succeeded" sous ~25 s.
!az containerapp job execution list \
    --name $WORKER_JOB --resource-group $RG \
    --query "[].{{name:name, status:properties.status, start:properties.startTime}}" \
    -o table

In [ ]:
# Logs applicatifs du worker via Log Analytics (KQL).
LAW_ID = az(f"az monitor log-analytics workspace show -n law-stage-{USER_ID} -g {RG} --query customerId -o tsv")
print(f"Workspace ID: {LAW_ID}\n")

QUERY = (
    "ContainerAppConsoleLogs_CL "
    "| where TimeGenerated > ago(10m) "
    "| where ContainerName_s == 'worker' "
    "| project TimeGenerated, Log_s "
    "| order by TimeGenerated desc "
    "| take 20"
)
!az monitor log-analytics query --workspace "{LAW_ID}" --analytics-query "{QUERY}" -o tsv

## 9. 🧹 Nettoyage ciblé

⚠️ **Ne supprime JAMAIS le Resource Group lui-même** (il appartient à ton équipe).

Pour tout détruire sauf le RG, **décommente les lignes qui t'intéressent** dans la cellule suivante.

💡 **Astuce coûts** : si tu veux juste "mettre en pause" sans tout supprimer, scale le `web` à 0 replica :

```bash
az containerapp update -n web-stage-<id> -g rg-stage-<id> --min-replicas 0 --max-replicas 0
```

Le worker, lui, est déjà à 0 quand la queue est vide (KEDA).

In [ ]:
# Décommentez pour exécuter

# # Apps + Job (libère les coûts ACA)
# !az containerapp delete --name $WEB_APP --resource-group $RG --yes
# !az containerapp job delete --name $WORKER_JOB --resource-group $RG --yes
# !az containerapp env delete --name $ACA_ENV --resource-group $RG --yes

# # Foundry (projet, puis ressource)
# !az cognitiveservices account project delete --name $FOUNDRY_NAME --resource-group $RG --project-name $FOUNDRY_PROJECT
# !az cognitiveservices account delete --name $FOUNDRY_NAME --resource-group $RG

# # Cosmos
# !az cosmosdb delete --name $COSMOS_NAME --resource-group $RG --yes

# # Storage + ACR + observabilité
# !az storage account delete --name $STORAGE_NAME --resource-group $RG --yes
# !az acr delete --name $ACR_NAME --resource-group $RG --yes
# !az monitor app-insights component delete --app appi-stage-$USER_ID --resource-group $RG
# !az monitor log-analytics workspace delete --workspace-name law-stage-$USER_ID --resource-group $RG --yes --force true

print(f"⚠️  Resource Group préservé : {RG}")
print("    Décommentez les lignes ci-dessus pour supprimer les ressources individuellement.")

## Récap

Tu as :
- Déployé toute l'infra en **2 passes Bicep** (base puis apps).
- Créé un **projet Foundry** + déployé le modèle `gpt-5-mini`.
- Buildé et poussé **2 images Docker** dans ton ACR managed.
- Déployé un **Container App** (`web`, ingress public) et un **Container App Job** event-driven KEDA (`worker`).
- Câblé toutes les **Managed Identities + RBAC** (Azure RBAC pour Storage/ACR/Foundry, RBAC natif Cosmos pour le data plane).
- Testé le pipeline upload → OCR → LLM → Cosmos de bout en bout.

## Schema architecture

Ouvre [docs/architecture.drawio](docs/architecture.drawio) pour voir le diagramme complet (extension VS Code *Draw.io Integration*, ou import sur https://app.diagrams.net).

## Pour aller plus loin

- **Réseau** : remplacer le public-access par VNet + Private Endpoints + App Gateway. Cf. ADR future dans `DESIGN.md`.

- **Auth utilisateurs** : ajouter Entra External ID (clients) + Entra ID (admins). Aujourd'hui, l'admin UI est en accès libre — à NE PAS faire en prod.- **Évaluations Foundry** : voir notebook 06 du parcours.

- **CRUD clients** : finaliser la page `/admin/clients`.- **CI/CD** : pipeline GitHub Actions qui rebuilde + redeploy sur push.